[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/02_Vision_Language_Models/01_clip_from_scratch/01_clip_from_scratch.ipynb)

# 01. CLIP from Scratch

**CLIP (Contrastive Language-Image Pre-training)** is the foundation of modern multimodal AI.

**This notebook covers:**
- CLIP architecture — built piece by piece
- Contrastive loss (InfoNCE) — visualized step by step
- Training CLIP on synthetic data (CPU-friendly)
- Visualizing the learned embedding space

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/02_Vision_Language_Models/01_clip_from_scratch")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from utils.visualization import *
from utils.helpers import *

set_style()
device = get_device()

## 1. CLIP Architecture Diagram

![CLIP Architecture — Radford et al. (2021)](../../assets/paper_figures/clip_overview.png)
*Source: Radford et al. (2021) — Learning Transferable Visual Models From Natural Language Supervision*

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('CLIP Architecture', fontsize=20, fontweight='bold', pad=20)

# Image side
draw_architecture_block(ax, 3, 9, 3, 0.7, 'Image', '#E74C3C')
draw_architecture_block(ax, 3, 7.5, 3, 0.9, 'Image Encoder\n(ViT or ResNet)', '#E74C3C')
draw_architecture_block(ax, 3, 5.8, 3, 0.7, 'Image Embedding\n[B, d_img]', '#C0392B')
draw_architecture_block(ax, 3, 4.3, 3, 0.7, 'Projection\nLinear(d_img, d)', '#C0392B')

# Text side
draw_architecture_block(ax, 11, 9, 3, 0.7, 'Text', '#3498DB')
draw_architecture_block(ax, 11, 7.5, 3, 0.9, 'Text Encoder\n(Transformer)', '#3498DB')
draw_architecture_block(ax, 11, 5.8, 3, 0.7, 'Text Embedding\n[B, d_txt]', '#2980B9')
draw_architecture_block(ax, 11, 4.3, 3, 0.7, 'Projection\nLinear(d_txt, d)', '#2980B9')

# L2 normalize
draw_architecture_block(ax, 3, 3.0, 2.5, 0.5, 'L2 Normalize', '#F39C12')
draw_architecture_block(ax, 11, 3.0, 2.5, 0.5, 'L2 Normalize', '#F39C12')

# Similarity matrix
draw_architecture_block(ax, 7, 1.5, 6, 1.2, 'Cosine Similarity Matrix / Temperature\n+ Contrastive Loss (InfoNCE)', '#9B59B6', fontsize=11)

# Arrows
for y_start, y_end in [(8.6, 8.0), (7.0, 6.2), (5.4, 4.7), (3.9, 3.3)]:
    draw_arrow(ax, (3, y_start), (3, y_end))
    draw_arrow(ax, (11, y_start), (11, y_end))

draw_arrow(ax, (3, 2.7), (5, 2.0))
draw_arrow(ax, (11, 2.7), (9, 2.0))

plt.tight_layout()
plt.savefig('../assets/clip_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

![CLIP Architecture — Radford et al. (2021)](../assets/paper_figure_clip_algorithm.png)

*Source: Radford et al. (2021) — "Learning Transferable Visual Models From Natural Language Supervision" — [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)*

## 📊 CLIP Variants: SigLIP & ALIGN

### SigLIP (Zhai et al., 2023)
**Paper:** [arXiv:2303.15343](https://arxiv.org/abs/2303.15343) — *Sigmoid Loss for Language Image Pre-Training*

SigLIP replaces softmax cross-entropy with a **sigmoid pairwise loss**:

$$
\mathcal{L}_{SigLIP} = -\frac{1}{N^2}\sum_{i=1}^{N}\sum_{j=1}^{N} \log \sigma\left(z_{ij} \cdot (-1)^{\mathbb{1}[i \neq j]} \cdot (t \cdot s_{ij} + b)\right)
$$

where $z_{ij} = 1$ if $i = j$ (positive pair), $z_{ij} = -1$ otherwise, and $s_{ij} = \text{sim}(I_i, T_j)$.

**Key advantages over CLIP:**
| Property | CLIP (Softmax) | SigLIP (Sigmoid) |
|----------|---------------|------------------|
| Loss type | Cross-entropy over N classes | Binary cross-entropy per pair |
| Negative samples | All N-1 in batch | All N² pairs |
| Batch size sensitivity | Needs very large batches (32K) | Works well with smaller batches |
| Global communication | Requires all-gather across GPUs | **No cross-device communication** |
| Memory scaling | $O(N)$ per sample | $O(1)$ per pair |

### ALIGN (Jia et al., 2021)
**Paper:** [arXiv:2102.05918](https://arxiv.org/abs/2102.05918) — *Scaling Up Visual and Vision-Language Representation Learning With Noisy Text Supervision*

ALIGN's key insight: **Scale data, not curation quality**:
- **CLIP:** 400M curated image-text pairs (WIT)
- **ALIGN:** 1.8B **noisy** alt-text pairs from web (no manual filtering!)
- Uses EfficientNet (not ViT) as vision encoder
- Same contrastive loss as CLIP

Result: Noisy data at scale > curated data at smaller scale!

![CLIP / Contrastive Alignment — Radford et al. (2021)](../assets/paper_figure_clip_algorithm.png)

*Source: Radford et al. (2021) — CLIP — [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)*

*See also: SigLIP — Zhai et al. (2023) — [arXiv:2303.15343](https://arxiv.org/abs/2303.15343); ALIGN — Jia et al. (2021) — [arXiv:2102.05918](https://arxiv.org/abs/2102.05918)*

## 2. Build CLIP Step by Step

In [ ]:
class CLIPImageEncoder(nn.Module):
    """Small ViT-style image encoder for CLIP."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 embed_dim=128, n_heads=4, n_layers=3):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # [B, N, D]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.transformer(x)
        return self.norm(x[:, 0])  # [CLS] output


class CLIPTextEncoder(nn.Module):
    """Small transformer text encoder for CLIP."""
    def __init__(self, vocab_size=5000, embed_dim=128, max_len=32,
                 n_heads=4, n_layers=3):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(x) + self.pos_embed(pos)
        x = self.transformer(x)
        return self.norm(x[:, 0])  # [CLS] output


class CLIP(nn.Module):
    """Full CLIP model with contrastive learning."""
    def __init__(self, embed_dim=128, projection_dim=64):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(embed_dim=embed_dim)
        self.text_encoder = CLIPTextEncoder(embed_dim=embed_dim)
        
        # Projection heads (map to shared space)
        self.image_proj = nn.Linear(embed_dim, projection_dim)
        self.text_proj = nn.Linear(embed_dim, projection_dim)
        
        # Learnable temperature
        self.temperature = nn.Parameter(torch.ones(1) * np.log(1 / 0.07))

    def encode_image(self, images):
        features = self.image_encoder(images)
        projected = self.image_proj(features)
        return F.normalize(projected, dim=-1)

    def encode_text(self, text_ids):
        features = self.text_encoder(text_ids)
        projected = self.text_proj(features)
        return F.normalize(projected, dim=-1)

    def forward(self, images, text_ids):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(text_ids)
        
        # Cosine similarity scaled by temperature
        logit_scale = self.temperature.exp()
        logits_per_image = logit_scale * img_emb @ txt_emb.T
        logits_per_text = logits_per_image.T
        
        return logits_per_image, logits_per_text, img_emb, txt_emb


model = CLIP(embed_dim=128, projection_dim=64)
count_parameters(model)

## 3. The Contrastive Loss (InfoNCE) — Visualized

**Key idea:** In a batch of N image-text pairs:
- The N diagonal entries are **positive pairs** (matching)
- The N²-N off-diagonal entries are **negative pairs** (non-matching)
- Loss = push positives together, push negatives apart

### Gradient Analysis of InfoNCE

For image $i$ in a batch, the image-to-text InfoNCE loss is:

$$\mathcal{L}_i = -\log \frac{\exp(s_{ii}/\tau)}{\sum_{j=1}^{N} \exp(s_{ij}/\tau)}$$

where $s_{ij} = \text{sim}(v_i, t_j)$ is the cosine similarity between image $i$ and text $j$, and $p_{ij} = \text{softmax}_j(s_{ij}/\tau)$ is the predicted probability that image $i$ matches text $j$.

Taking the derivative w.r.t. each similarity score:

$$\frac{\partial \mathcal{L}_i}{\partial s_{ii}} = p_{ii} - 1 \qquad \text{(positive pair)}$$

$$\frac{\partial \mathcal{L}_i}{\partial s_{ij}} = p_{ij} \qquad \text{(negative pair, } j \neq i \text{)}$$

**Interpretation:**

| Pair type | Gradient sign | Effect |
|-----------|--------------|--------|
| Positive $(i,i)$ | $p_{ii} - 1 \leq 0$ | **Increase** $s_{ii}$ (push similarity up) |
| Negative $(i,j)$ | $p_{ij} \geq 0$ | **Decrease** $s_{ij}$ (push similarity down) |

**Key insight:** The gradient magnitude is largest when the model is **most confused** — when $p_{ii}$ is low (positive pair looks like a negative), $|p_{ii} - 1|$ is large, producing a strong learning signal. When $p_{ii} \approx 1$ (well-learned pair), the gradient vanishes. The model naturally focuses its capacity on the pairs it gets wrong.

In [ ]:
def clip_loss(logits_per_image, logits_per_text):
    """Symmetric contrastive loss (InfoNCE)."""
    batch_size = logits_per_image.shape[0]
    labels = torch.arange(batch_size, device=logits_per_image.device)
    
    # Image-to-text: which text matches this image?
    loss_i2t = F.cross_entropy(logits_per_image, labels)
    # Text-to-image: which image matches this text?
    loss_t2i = F.cross_entropy(logits_per_text, labels)
    
    return (loss_i2t + loss_t2i) / 2


# Visualize the loss computation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('CLIP Contrastive Loss — Step by Step', fontsize=16, fontweight='bold')

B = 5
img_emb = F.normalize(torch.randn(B, 64), dim=-1)
txt_emb = F.normalize(torch.randn(B, 64), dim=-1)
sim = (img_emb @ txt_emb.T).detach().numpy()

# Step 1: Raw similarity
ax = axes[0]
im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Step 1: Cosine Similarity')
ax.set_xlabel('Text')
ax.set_ylabel('Image')
for i in range(B):
    for j in range(B):
        color = 'white' if abs(sim[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# Step 2: Labels (diagonal = positive)
ax = axes[1]
target = np.eye(B)
ax.imshow(target, cmap='Greens')
ax.set_title('Step 2: Target\n(diagonal = matching pairs)')
ax.set_xlabel('Text')
ax.set_ylabel('Image')
for i in range(B):
    for j in range(B):
        label = '✓ match' if i == j else '✗'
        color = 'white' if i == j else 'gray'
        ax.text(j, i, label, ha='center', va='center', fontsize=9, color=color)

# Step 3: After training (what we want)
ax = axes[2]
ideal = np.eye(B) * 0.95 + (1 - np.eye(B)) * (-0.3) + np.random.randn(B, B) * 0.05
np.fill_diagonal(ideal, np.random.uniform(0.85, 0.95, B))
im = ax.imshow(ideal, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Step 3: After Training\n(diagonal HIGH, rest LOW)')
ax.set_xlabel('Text')
ax.set_ylabel('Image')
for i in range(B):
    for j in range(B):
        color = 'white' if abs(ideal[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{ideal[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('../assets/clip_loss_visual.png', dpi=150, bbox_inches='tight')
plt.show()

### Numerical InfoNCE Example

Let's compute InfoNCE by hand for a batch of **3 image-text pairs** with temperature $\tau = 0.07$.

Recall: logits $= s_{ij} / \tau$ where $s_{ij}$ is cosine similarity.

#### Case 1: Well-learned pair

Similarities (diagonal positive highlighted): $[0.8,\; 0.1,\; -0.2]$ for pair $(i,i)$

| Step | Computation | Result |
|------|-------------|--------|
| Logits | $[0.8/0.07,\; 0.1/0.07,\; -0.2/0.07]$ | $[11.4,\; 1.4,\; -2.9]$ |
| Softmax | $\text{softmax}(\text{logits})$ | $[0.9998,\; 0.0002,\; 0.0000]$ |
| Loss | $-\log(0.9998)$ | $\approx 0.0002$ |

The model is **highly confident** about the correct pairing — gradient magnitude is tiny (see Gradient Analysis above: $p_{ii} - 1 \approx -0.0002$).

#### Case 2: Confused model — high loss drives learning

Similarities: $[0.3,\; 0.25,\; 0.2]$ — all pairs look equally similar!

| Step | Computation | Result |
|------|-------------|--------|
| Logits | $[4.3,\; 3.6,\; 2.9]$ | |
| Softmax | | $[0.56,\; 0.28,\; 0.14]$ |
| Loss | $-\log(0.56)$ | $\approx 0.58$ |

**High loss** $\Rightarrow$ **strong gradient** ($p_{ii} - 1 = -0.44$) $\Rightarrow$ the model receives a large signal to **separate** the positive pair from the negatives. This is exactly why contrastive learning works: confused batches produce the strongest learning signal.

## 4. Train CLIP on Synthetic Data (CPU-Friendly)

In [ ]:
# Create synthetic dataset
images, texts, labels = create_synthetic_image_text_pairs(n_samples=200, img_size=32, n_classes=5)

# Simple tokenizer (word-level)
all_words = set()
for t in texts:
    all_words.update(t.lower().split())
word2id = {w: i+2 for i, w in enumerate(sorted(all_words))}
word2id['[PAD]'] = 0
word2id['[CLS]'] = 1

def tokenize(text, max_len=16):
    ids = [word2id['[CLS]']] + [word2id.get(w, 0) for w in text.lower().split()]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return torch.tensor(ids)

# Create DataLoader
class CLIPDataset(Dataset):
    def __init__(self, images, texts):
        self.images = images
        self.texts = texts
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.images[idx], tokenize(self.texts[idx])

dataset = CLIPDataset(images, texts)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset: {len(dataset)} image-text pairs")
print(f"Vocab: {len(word2id)} words")
print(f"Sample: '{texts[0]}' → {tokenize(texts[0]).tolist()[:8]}...")

In [ ]:
# Train CLIP!
model = CLIP(embed_dim=128, projection_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

losses = []
n_epochs = 30

for epoch in range(n_epochs):
    epoch_loss = 0
    for images_batch, text_batch in loader:
        images_batch = images_batch.to(device)
        text_batch = text_batch.to(device)
        
        logits_i2t, logits_t2i, _, _ = model(images_batch, text_batch)
        loss = clip_loss(logits_i2t, logits_t2i)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f}")

# Plot training curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, linewidth=2, color='#9B59B6')
ax.set_xlabel('Epoch')
ax.set_ylabel('Contrastive Loss')
ax.set_title('CLIP Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Zero-Shot Classification

One of CLIP's most powerful capabilities requires **no additional training** — only inference over precomputed text embeddings.

**Setup:** Given class names $C = \{\text{"cat"}, \text{"dog"}, \text{"car"}\}$, encode each as a prompt:

$$t_k = \text{encode\_text}(\text{"a photo of a \{class\_k\}"})$$

**Inference:** For any image embedding $v$, predict the class via temperature-scaled softmax over cosine similarities:

$$\hat{k} = \arg\max_k \; \frac{\exp(\text{sim}(v, t_k) / \tau)}{\sum_j \exp(\text{sim}(v, t_j) / \tau)}$$

This is identical to the InfoNCE softmax from training — but now the "batch" of text embeddings is the set of class prompts, and we only have a single image query.

**Why it works:** During contrastive pre-training, CLIP learns that "a photo of a dog" should be near dog images and far from cat images. At inference, we simply enumerate candidate descriptions and pick the closest one.

**No fine-tuning needed** — just encode class names once, cache the embeddings, and classify any new image in a single forward pass.

In [ ]:
# Visualize the learned similarity matrix
model.eval()
with torch.no_grad():
    test_imgs = torch.stack(images[:10]).to(device)
    test_txts = torch.stack([tokenize(t) for t in texts[:10]]).to(device)
    
    img_emb = model.encode_image(test_imgs)
    txt_emb = model.encode_text(test_txts)

fig = plot_similarity_matrix(
    img_emb, txt_emb,
    labels_a=[f'img_{i}' for i in range(10)],
    labels_b=[t[:15] for t in texts[:10]],
    title='CLIP Learned Similarity (after training)'
)
plt.savefig('../assets/clip_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Diagonal should be brighter (matching pairs have higher similarity)")

In [ ]:
# ============================================================
#  Example 1: Zero-Shot Classification — Worked Example
# ============================================================

print("=" * 65)
print("  ZERO-SHOT CLASSIFICATION: Step-by-Step")
print("=" * 65)

# Use our trained model for zero-shot classification
model.eval()

# Define class text prompts
class_names = ['red circle', 'blue square', 'green triangle', 'yellow star', 'white diamond']
class_prompts = [tokenize(f"a photo of a {name}") for name in class_names]
class_text_ids = torch.stack(class_prompts).to(device)

# Encode all class descriptions (do this ONCE, cache for inference)
with torch.no_grad():
    class_text_emb = model.encode_text(class_text_ids)  # [5, 64]
    print(f"Class embeddings shape: {class_text_emb.shape}")
    print(f"Class embeddings are L2-normalized: norm = {class_text_emb.norm(dim=-1)}")

# Classify test images
n_test = 50
test_imgs = torch.stack(images[:n_test]).to(device)
test_labels_true = torch.tensor(labels[:n_test])

with torch.no_grad():
    test_img_emb = model.encode_image(test_imgs)  # [50, 64]
    
    # Similarity matrix: each image vs each class prompt
    similarities = test_img_emb @ class_text_emb.T  # [50, 5]
    
    # Scale by temperature
    logit_scale = model.temperature.exp()
    logits = logit_scale * similarities
    
    # Predict: argmax over class dimension
    probs = torch.softmax(logits, dim=-1)
    predictions = probs.argmax(dim=-1)

# Show step-by-step for first 5 images
print(f"\nStep-by-step for first 5 test images:")
print(f"{'Image':>6} | {'True':>12} | {'Predicted':>12} | {'Confidence':>10} | {'All probs'}")
print("-" * 75)
for i in range(5):
    true_cls = class_names[test_labels_true[i]] if test_labels_true[i] < len(class_names) else f"class_{test_labels_true[i]}"
    pred_cls = class_names[predictions[i]]
    conf = probs[i, predictions[i]].item()
    all_p = " ".join(f"{p:.3f}" for p in probs[i].cpu().numpy())
    match = "✅" if predictions[i].item() == test_labels_true[i].item() else "❌"
    print(f"  {i:>4} | {true_cls:>12} | {pred_cls:>12} | {conf:>10.4f} | [{all_p}] {match}")

# Overall accuracy
correct = (predictions.cpu() == test_labels_true).sum().item()
accuracy = correct / n_test * 100
print(f"\n🎯 Zero-Shot Accuracy: {accuracy:.1f}% ({correct}/{n_test})")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Similarity heatmap
ax = axes[0]
sim_to_show = similarities[:10].cpu().numpy()
im = ax.imshow(sim_to_show, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(5))
ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Test Image Index')
ax.set_title('Cosine Similarity\n(Image vs Class Prompts)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax)

# Confusion matrix
ax = axes[1]
conf_matrix = np.zeros((5, 5), dtype=int)
for t, p in zip(test_labels_true.numpy(), predictions.cpu().numpy()):
    if t < 5 and p < 5:
        conf_matrix[t][p] += 1
im = ax.imshow(conf_matrix, cmap='Blues')
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(conf_matrix[i][j]), ha='center', va='center', fontsize=11,
                color='white' if conf_matrix[i][j] > 3 else 'black')
ax.set_title(f'Confusion Matrix\n(Accuracy: {accuracy:.1f}%)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax)

# Per-class accuracy
ax = axes[2]
per_class_acc = []
for c in range(5):
    mask = test_labels_true == c
    if mask.sum() > 0:
        per_class_acc.append((predictions.cpu()[mask] == c).float().mean().item() * 100)
    else:
        per_class_acc.append(0)
ax.bar(class_names, per_class_acc, color=['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6'], alpha=0.8)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Zero-Shot Accuracy', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
for i, acc in enumerate(per_class_acc):
    ax.text(i, acc + 2, f'{acc:.0f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/zero_shot_classification.png', dpi=150, bbox_inches='tight')
plt.show()

### Example 2: Image-Text Retrieval — Cross-Modal Search

CLIP's shared embedding space enables **bidirectional retrieval** without any task-specific training:

- **Image → Text (I2T):** Given an image, find the most relevant caption
- **Text → Image (T2I):** Given a text query, find the most relevant image

The retrieval metric is **Recall@K** — the fraction of queries where the correct match appears in the top-K results:

$$\text{R@K} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}\left[\text{rank}(i) \leq K\right]$$

| Metric | What it measures | Good value |
|--------|-----------------|-----------|
| R@1 | Correct match is the top result | >50% |
| R@5 | Correct match in top 5 | >80% |
| R@10 | Correct match in top 10 | >90% |

In [ ]:
# ============================================================
#  Example 2: Image-Text Retrieval with Recall@K
# ============================================================

print("=" * 65)
print("  IMAGE-TEXT RETRIEVAL: Recall@K Evaluation")
print("=" * 65)

model.eval()

# Use full dataset for retrieval evaluation
N_eval = min(100, len(images))
eval_imgs = torch.stack(images[:N_eval]).to(device)
eval_txts = torch.stack([tokenize(t) for t in texts[:N_eval]]).to(device)

with torch.no_grad():
    img_emb = model.encode_image(eval_imgs)  # [N, 64]
    txt_emb = model.encode_text(eval_txts)  # [N, 64]
    
    # Full similarity matrix
    sim_matrix = img_emb @ txt_emb.T  # [N, N]

# Compute Recall@K
def compute_recall_at_k(sim_matrix, ks=[1, 5, 10]):
    """Compute recall@k for image-to-text and text-to-image retrieval."""
    N = sim_matrix.shape[0]
    results = {}
    
    for direction, mat in [("I2T", sim_matrix), ("T2I", sim_matrix.T)]:
        for k in ks:
            # Get top-k indices for each query
            _, topk_indices = mat.topk(k, dim=1)
            
            # Check if ground truth (diagonal) is in top-k
            gt = torch.arange(N, device=mat.device).unsqueeze(1)
            recall = (topk_indices == gt).any(dim=1).float().mean().item() * 100
            results[f"{direction}_R@{k}"] = recall
    
    return results

recalls = compute_recall_at_k(sim_matrix)

print(f"\nRetrieval Results (N={N_eval} pairs):")
print(f"{'Metric':<12} {'Score':>8}")
print("-" * 22)
for metric, score in sorted(recalls.items()):
    print(f"  {metric:<10} {score:>7.1f}%")

# Show retrieval examples
print(f"\n📋 Image→Text Retrieval Examples:")
for i in range(5):
    top3_idx = sim_matrix[i].topk(3).indices.cpu().numpy()
    print(f"  Image {i} → Top-3 texts: {top3_idx}  "
          f"(correct={i in top3_idx}{'✅' if i in top3_idx else '❌'})")
    for rank, idx in enumerate(top3_idx):
        sim_val = sim_matrix[i, idx].item()
        marker = " ← correct" if idx == i else ""
        print(f"    Rank {rank+1}: text_{idx} (sim={sim_val:.3f}){marker}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recall@K bar chart
ax = axes[0]
ks = [1, 5, 10]
i2t_recalls = [recalls[f"I2T_R@{k}"] for k in ks]
t2i_recalls = [recalls[f"T2I_R@{k}"] for k in ks]
x = np.arange(len(ks))
width = 0.35
ax.bar(x - width/2, i2t_recalls, width, label='Image→Text', color='#E74C3C', alpha=0.8)
ax.bar(x + width/2, t2i_recalls, width, label='Text→Image', color='#3498DB', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'R@{k}' for k in ks])
ax.set_ylabel('Recall (%)')
ax.set_title('Retrieval Performance (Recall@K)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 105)
for i, (v1, v2) in enumerate(zip(i2t_recalls, t2i_recalls)):
    ax.text(i - width/2, v1 + 1.5, f'{v1:.0f}%', ha='center', fontsize=9)
    ax.text(i + width/2, v2 + 1.5, f'{v2:.0f}%', ha='center', fontsize=9)

# Similarity matrix heatmap
ax = axes[1]
sim_np = sim_matrix[:15, :15].cpu().numpy()
im = ax.imshow(sim_np, cmap='YlOrRd', aspect='equal')
ax.set_title('Similarity Matrix (first 15)\n(bright diagonal = good retrieval)', fontsize=12, fontweight='bold')
ax.set_xlabel('Text Index')
ax.set_ylabel('Image Index')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('../assets/clip_retrieval.png', dpi=150, bbox_inches='tight')
plt.show()

### Example 3: Real-World CLIP Applications

CLIP's zero-shot capability enables many applications without task-specific training:

| Application | How CLIP is Used | Industry |
|------------|-----------------|----------|
| **Visual Search** | User uploads photo → CLIP encodes → finds similar products by text description | E-commerce (Amazon, Pinterest) |
| **Content Moderation** | Encode "violent content" / "nudity" prompts → score all images → flag high-similarity | Social media (Meta, TikTok) |
| **Medical Imaging** | "X-ray showing pneumonia" prompt → retrieve matching X-rays from database | Healthcare |
| **Autonomous Driving** | "Pedestrian crossing road" → score camera frames → assist detection | Automotive |
| **Accessibility** | Score image against description candidates → auto-generate alt text | Web accessibility |
| **Robotics** | "Pick up the red cup" → CLIP scores visual observations → guide grasping | Robotics (CLIP-Fields) |

**Architecture Pattern:**
```
Offline: Encode all images/texts → store in vector database (FAISS, Pinecone)
Online:  Encode query → ANN search → return top-K results in <10ms
```

**Scaling:** CLIP embeddings are just 512-dim vectors. A million images = ~2GB of embeddings. With approximate nearest neighbor search (FAISS), retrieval over 1B images takes <100ms.

In [ ]:
# ============================================================
#  Example 3: CLIP Scaling — Batch Size and Performance
# ============================================================

print("=" * 65)
print("  CLIP SCALING ANALYSIS")
print("=" * 65)

# Show how batch size affects negative examples and loss
print("\n1. Batch Size → Number of Negatives:")
print(f"   {'Batch Size':>12} | {'Negatives':>10} | {'Random Baseline':>15} | {'GPU Memory (ViT-B/32)':>22}")
print("   " + "-" * 65)

for bs in [32, 128, 512, 2048, 8192, 32768]:
    negatives = bs - 1
    baseline_loss = np.log(bs)
    # Rough memory estimate: 2 * batch * seq * d * 4 bytes
    mem_gb = 2 * bs * 197 * 768 * 4 / 1e9
    print(f"   {bs:>12,} | {negatives:>10,} | ln({bs}) = {baseline_loss:>7.2f} | ~{mem_gb:.1f} GB")

# Simulate how loss changes with more negatives
print(f"\n2. Effect of Batch Size on Contrastive Learning:")
batch_sizes = [4, 8, 16, 32, 64, 128, 256]
losses_by_bs = []

for bs in batch_sizes:
    # Create synthetic similarity matrix with one strong positive
    sim = torch.randn(bs, bs) * 0.3
    sim.fill_diagonal_(0.8)  # positive pairs have sim=0.8
    
    scaled = sim / 0.07
    labels = torch.arange(bs)
    loss = F.cross_entropy(scaled, labels).item()
    losses_by_bs.append(loss)
    
    # P(correct) for the positive pair
    probs = torch.softmax(scaled[0], dim=0)
    print(f"   BS={bs:>4}: Loss={loss:.4f}, P(correct)={probs[0]:.4f}, "
          f"Best negative={probs[1:].max():.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(batch_sizes, losses_by_bs, 'o-', color='#E74C3C', linewidth=2, markersize=8, label='Actual loss')
ax.plot(batch_sizes, [np.log(bs) for bs in batch_sizes], 's--', color='gray', alpha=0.5, label='Random baseline ln(N)')
ax.set_xlabel('Batch Size')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Loss vs Batch Size\n(more negatives = harder task)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xscale('log', base=2)

ax = axes[1]
# Embedding space quality vs batch size
qualities = [max(0, 1 - l/np.log(bs)) for l, bs in zip(losses_by_bs, batch_sizes)]
ax.bar([str(bs) for bs in batch_sizes], qualities, color='#2ECC71', alpha=0.8)
ax.set_xlabel('Batch Size')
ax.set_ylabel('Alignment Quality (1 - loss/baseline)')
ax.set_title('Embedding Quality vs Batch Size\n(larger batches = better alignment)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('../assets/clip_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

### Temperature Learning Dynamics

CLIP does not treat temperature as a fixed hyperparameter — it is a **learnable logit scale** initialized so that $\tau = \exp(\log(1/0.07)) \approx 14.3$ at the start of training (equivalently, the raw similarity is divided by $0.07$ before softmax).

**Why learn temperature?** Early in training, embeddings are poorly aligned; a softer distribution (lower effective temperature) prevents the model from over-committing to wrong pairings. As alignment improves, $\tau$ **increases**, sharpening the softmax and making the contrastive signal more decisive.

**Numerical stability:** In the original CLIP implementation, the logit scale is clipped:

$$\tau \leq 100$$

Without clipping, an unbounded temperature can produce extreme logits, causing softmax saturation (gradients vanish) or overflow in mixed-precision training.

**Intuition:** Temperature is the model's confidence dial — low $\tau$ = "I'm not sure yet, explore"; high $\tau$ = "I'm confident, push hard on the correct pairs."

## Key Takeaways

1. **CLIP = Image Encoder + Text Encoder + Contrastive Loss**
2. **InfoNCE loss** pulls matching pairs together, pushes non-matching apart
3. **Temperature** controls how "sharp" the similarity distribution is
4. After training, you can do **zero-shot classification** by comparing image embeddings to text embeddings
5. CLIP's shared space enables many downstream tasks without retraining

---
**Next:** `02_image_captioning.ipynb` - Generate text from images

---

## 📚 References & Further Reading

### Papers
- **Learning Transferable Visual Models From Natural Language Supervision (CLIP)** — Radford et al., 2021 — [arXiv:2103.00020](https://arxiv.org/abs/2103.00020) — The original CLIP paper
- **Reproducible Scaling Laws for Contrastive Language-Image Learning (OpenCLIP)** — Cherti et al., 2023 — [arXiv:2212.07143](https://arxiv.org/abs/2212.07143) — Open-source CLIP training at scale
- **ALIGN: Scaling Up Visual and Vision-Language Representation Learning** — Jia et al., 2021 — [arXiv:2102.05918](https://arxiv.org/abs/2102.05918) — Google's CLIP-like model with noisy data
- **Sigmoid Loss for Language Image Pre-Training (SigLIP)** — Zhai et al., 2023 — [arXiv:2303.15343](https://arxiv.org/abs/2303.15343) — Sigmoid alternative to InfoNCE
- **FLIP: Scaling Language-Image Pre-training via Masking** — Li et al., 2023 — [arXiv:2212.00794](https://arxiv.org/abs/2212.00794) — Efficient CLIP training with masking
- **Representation Learning with Contrastive Predictive Coding** — van den Oord et al., 2018 — [arXiv:1807.03748](https://arxiv.org/abs/1807.03748) — InfoNCE loss derivation

### Blog Posts & Cheat Sheets
- 🔗 [OpenAI CLIP Blog Post](https://openai.com/research/clip) — Original announcement with interactive demos
- 🔗 [The Illustrated CLIP](https://blog.roboflow.com/openai-clip/) — Roboflow — Visual architecture walkthrough
- 🔗 [Lilian Weng: Contrastive Representation Learning](https://lilianweng.github.io/posts/2021-05-31-contrastive/) — Deep dive into contrastive losses
- 🔗 [OpenCLIP GitHub](https://github.com/mlfoundations/open_clip) — Open-source CLIP implementation with pretrained weights
- 🔗 [CLIP Retrieval](https://rom1504.github.io/clip-retrieval/) — Interactive demo: search LAION-5B with text queries
- 🔗 [Hugging Face CLIP Guide](https://huggingface.co/docs/transformers/model_doc/clip) — How to use CLIP in 5 lines of code
- 🔗 [FAISS: A Library for Efficient Similarity Search](https://github.com/facebookresearch/faiss) — Scale CLIP retrieval to billions of images